# Part 2 — The dataset and its contracts

Every accuracy number in the current record is measured on one public dataset. The
project treats it less as a download than as a set of contracts: what the files
contain, which jets may ever touch a reported number, how constituents are ordered
before the model sees them, and which preprocessing must travel with a checkpoint.

Run this notebook from the repository root.

## What the dataset is

The HLS4ML LHC Jet dataset in its 150-particle form, Zenodo record 3602260 — the public
benchmark of the hls4ml program (arXiv:1804.06913, arXiv:1908.05318). The task is
five-way classification of simulated jets: gluon (g), light quark (q), W, Z, and top (t).

Using the community's own benchmark is deliberate. It makes the project's numbers
comparable to published taggers instead of only to itself (`RESEARCH.md` §4).

### What a file contains

- `jetConstituentList`, shape (10000, 150, 16): up to 150 zero-padded particles per
  jet, 16 features each (px, py, pz, e, erel, pt, ptrel, eta, etarel, etarot, phi,
  phirel, phirot, deltaR, costheta, costhetarel).
- `jets`, shape (10000, 59): jet-level kinematics and substructure, whose last six
  columns are the one-hot targets (the five classes plus a never-used `j_undef`).
- Calorimeter-image views (`jetImage`, `jetImageECAL`, `jetImageHCAL`), unused by every
  model in this project.

### Measured, not read off the landing page

| Property | Value |
|---|---|
| Validation archive | `hls4ml_LHCjet_150p_val.tar.gz`, 1.14 GB |
| Files and jets | 26 HDF5 files of 10,000 jets = 260,000 jets, not the 270,000 the file numbering suggests: one chunk (`jetImage_9_150p_40000_50000.h5`) is absent from the tarball, verified with `tar -tzf` |
| Class counts (g / q / W / Z / t) | 52,404 / 50,468 / 52,235 / 52,298 / 52,595 — essentially uniform, quark about 4% lighter |
| Constituent occupancy | mean 49.4, median 46, range 5–150; 95% of jets below 89 |
| Per-feature scale spread | standard deviations span a ~2,979x range, GeV-scale momenta against order-one angles |
| Jet kinematics | median jet pT 1022.1 GeV (range 159.3–3156.7), median jet mass 86.9 GeV |

Every figure above was measured directly from the local files on 2026-08-01 with a
re-runnable inspection script and recorded in `DATASET.md`, which is the source for
this section.

## Two splits, two AUCs

The dataset ships with its own train/validation division, and the two splits get fixed,
non-negotiable roles.

- The 260,000-jet val split is the **ROC-test set**, on which every accuracy number of
  record (ROC-test macro one-vs-rest AUC, n = 260,000) is computed. It is used for
  nothing else: no training, no early stopping, no calibration, no hyperparameter
  selection ever touches it (`RESEARCH.md` §4).
- Training's own validation split is carved from the train tarball: the training stage
  shuffles with the run's seed and holds out 20% (`validation_split = 0.20`). The
  **validation macro-OvR AUC**, the epoch-by-epoch training monitor, is computed there.

Different data, different roles. They are never conflated in a table.

## Nobody downloads training data locally

There is no local copy of the training data and no shared cluster volume. Every pod
fetches the train tarball from Zenodo at the start of its run and gates on the
download's size before extracting, literally as follows (from the round-14 job
manifest):

```bash
# fetch HLS4ML LHC Jet 150p TRAIN split from Zenodo (~3 GB, PVC-free)
python -u -c 'import urllib.request; urllib.request.urlretrieve(
  "https://zenodo.org/records/3602260/files/hls4ml_LHCjet_150p_train.tar.gz?download=1",
  "/work/data/train.tar.gz")'
sz=$(stat -c %s /work/data/train.tar.gz)
[ "$sz" -gt 2500000000 ] || { echo "[fatal] train tarball too small"; exit 1; }
```

A truncated download would otherwise train on partial data without complaint. The
threshold sits safely below the real tarball size of 2,725,115,104 bytes, measured
2026-07-07. The loader itself never downloads anything: it is a pure reader over a
directory of `.h5` files handed to it through an environment variable, which keeps the
data path identical between a pod and a local smoke test.

## Top-N constituents by pT

A jet arrives as up to 150 constituents; the model reads the N hardest. The loader
performs a *stable* argsort on negative constituent pT and truncates to the top N.

Stability matters. Ties in pT resolve the same way every time, so the model's input is
reproducible bit for bit between training, evaluation, and export, and the alignment
check that compares label arrays row-for-row against the stored evaluation arrays
depends on it. The physics rationale is that the hardest constituents carry most of the
discriminating substructure, and the measured occupancy above says the truncation
discards little.

In [ ]:
import numpy as np

# one fake jet in the file layout: 150 zero-padded constituents, 16 features
rng = np.random.default_rng(0)
jet = np.zeros((150, 16), dtype=np.float32)
n_real = 49                                   # about the measured mean occupancy
jet[:n_real] = rng.normal(size=(n_real, 16))
PT_COL = 5                                    # 'pt' in the canonical feature order
jet[:n_real, PT_COL] = rng.gamma(2.0, 20.0, size=n_real)

order = np.argsort(-jet[:, PT_COL], kind="stable")   # stable: ties resolve identically
top8 = jet[order][:8]

print("pT of the 8 hardest:", np.round(top8[:, PT_COL], 2))
print("descending:", bool(np.all(np.diff(top8[:, PT_COL]) <= 0)))
print("kept shape:", top8.shape)

The real loader is `load_eval_set` in `bnjettag/code/hgq2/bnhgq2/data.py`, which does
the same thing over a whole file batch with `np.take_along_axis`, and the training
loader replicates it line for line. Historically N = 10 with all 16 features, input
shape (10, 16); round 14 made N an experimental axis and swept N in {8, 16, 32, 64}
(`RESEARCH.md` §5‴).

### Features resolve by exact suffix

Feature selection, when a subset is used, happens *after* the sort and truncation, so
particle ordering is identical regardless of the feature set. Names resolve by exact
suffix match and the function raises unless each resolves uniquely: `pt` must not
silently match `ptrel`, nor `etarel` match `etarot`.

In [ ]:
src = open("code/bnhgq2/data.py").read()
print(src[src.index("def feature_indices"):src.index("def load_eval_set")].rstrip())

## Standardization is part of the model, not the data

From round 8 onward, inputs are z-scored per feature,

$$\hat{x} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}},$$

with $\mu$ and $\sigma$ computed on the training split only and written to
`input_std.json` beside the checkpoint at training time. Evaluation and hardware export
apply the same file.

Two reasons, both load-bearing.

- **Leakage.** Statistics fitted on train data only means the ROC-test set contributes
  nothing, not even its mean, to any reported number.
- **Contract.** The standardization is part of the model's input definition, exactly as
  much as the weights are. A checkpoint scored without its own `input_std.json` is
  scored wrong — quietly, and sometimes plausibly. That is why the file travels with the
  checkpoint everywhere instead of being recomputed ad hoc.

The measured ~2,979x per-feature scale spread is what makes this conditioning matter as
much as it does.

In [ ]:
src = open("code/bnhgq2/data.py").read()
print(src[src.index("def input_std_stats"):].rstrip())

## The three-feature restriction

Round 14 restricts the input to three features per constituent: pt, etarel, phirel. The
reason is the trigger itself — a Level-1 particle stream provides essentially the
kinematics of each candidate (transverse momentum and position relative to the jet
axis), not the sixteen derived quantities the offline dataset stores.

The convention follows Odagiu et al. (arXiv:2402.01876), the reference point for
constituent-based L1 taggers on this dataset, so round-14 results are input-matched to
that literature. `DATASET.md` records the corollary that cuts the other way: results
using all 16 features are *not* input-matched to those baselines, and any comparison
across the two input sets has to say so.

In [ ]:
import json

cfg = json.load(open("code/configs/r14-l1x3-n8-w1a8.json"))
arch = cfg["arch"]
print(cfg["name"], "| era", cfg["era"])
for k in ("n_part", "n_feat", "features", "input_std"):
    print(f"  {k:10s} {arch[k]}")

This config is one point of the round-14 sweep — the N = 8 arm. The rest of the sweep
is the same file with `n_part` set to 16, 32, or 64, generated rather than hand-edited;
part 5 covers config generation and pre-registration.

## One dataset, one era

The project migrated to this public dataset on 2026-07-01. Everything trained before
that date used a private two-class dataset and constitutes a separate era of the
record, archived wholesale under `archive/era1/` and never quoted beside a current
number. The firewall between eras is a results-discipline rule, and it is absolute: no
table, no figure, no sentence mixes the two.